In [2]:
import pandas as pd
import numpy as np

master = pd.read_csv("/Users/dholakiyan/Desktop/insider-threat-detection/data/processed/final_features.csv")
master['day'] = pd.to_datetime(master['day'])

print(master.shape)
print(master.columns.tolist())

(330452, 22)
['user', 'day', 'logon_count', 'device_count', 'file_count', 'email_count', 'http_count', 'logon_count_7day_avg', 'logon_count_deviation', 'device_count_7day_avg', 'device_count_deviation', 'file_count_7day_avg', 'file_count_deviation', 'email_count_7day_avg', 'email_count_deviation', 'http_count_7day_avg', 'http_count_deviation', 'after_hours_logon_count', 'is_first_usb_use', 'external_email_count', 'is_malicious_user', 'is_scenario_day']


### Select feature columns (exclude IDs and labels)

In [3]:
feature_cols = [
    'logon_count', 'device_count', 'file_count', 'email_count', 'http_count',
    'logon_count_deviation', 'device_count_deviation', 'file_count_deviation',
    'email_count_deviation', 'http_count_deviation',
    'after_hours_logon_count', 'external_email_count'
]

master['is_first_usb_use'] = master['is_first_usb_use'].astype(int)
feature_cols.append('is_first_usb_use')

print("Using", len(feature_cols), "features")

Using 13 features


### Handle NaNs from rolling window calculations
(The NaNs come from the first few days per user where there wasn't enough history yet for a 7-day rolling average — filling with 0 is a reasonable default meaning "no deviation detected yet.")

In [5]:
print(master[feature_cols].isna().sum())
master[feature_cols] = master[feature_cols].fillna(0)

logon_count                   0
device_count                  0
file_count                    0
email_count                   0
http_count                    0
logon_count_deviation      3000
device_count_deviation     3000
file_count_deviation       3000
email_count_deviation      3000
http_count_deviation       3000
after_hours_logon_count       0
external_email_count          0
is_first_usb_use              0
dtype: int64


### Chronological train/test split (reuse split date)

In [6]:
split_date = pd.Timestamp("2010-06-01")

train_df = master[master['day'] < split_date].copy()
test_df = master[master['day'] >= split_date].copy()

X_train = train_df[feature_cols]
X_test = test_df[feature_cols]

print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (105171, 13) Test: (225281, 13)


### Scale features (fit on train only)

In [7]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Train Isolation Forest

In [9]:
from sklearn.ensemble import IsolationForest

iso_forest = IsolationForest(
    n_estimators=200,
    contamination=0.02,
    random_state=42
)

iso_forest.fit(X_train_scaled)

test_df['anomaly_score'] = iso_forest.decision_function(X_test_scaled)
test_df['predicted_anomaly'] = iso_forest.predict(X_test_scaled)

print(test_df['predicted_anomaly'].value_counts())

predicted_anomaly
 1    221195
-1      4086
Name: count, dtype: int64


### The real test — does it catch known scenario days?

In [10]:
scenario_days = test_df[test_df['is_scenario_day'] == True]
normal_days = test_df[test_df['is_scenario_day'] == False]

flagged_scenario = (scenario_days['predicted_anomaly'] == -1).sum()
flagged_normal = (normal_days['predicted_anomaly'] == -1).sum()

print(f"Scenario days flagged as anomaly: {flagged_scenario} / {len(scenario_days)} ({100*flagged_scenario/len(scenario_days):.1f}%)")
print(f"Normal days flagged as anomaly: {flagged_normal} / {len(normal_days)} ({100*flagged_normal/len(normal_days):.1f}%)")

Scenario days flagged as anomaly: 82 / 1364 (6.0%)
Normal days flagged as anomaly: 4004 / 223917 (1.8%)


### Save the scored test set

In [11]:
test_df.to_csv("/Users/dholakiyan/Desktop/insider-threat-detection/data/processed/isolation_forest_results.csv", index=False)

### Save the trained model + scaler (first pickle files)

In [12]:
import pickle
import os

os.makedirs("../models", exist_ok=True)

with open("../models/isolation_forest.pkl", "wb") as f:
    pickle.dump(iso_forest, f)

with open("../models/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("Model and scaler saved.")

Model and scaler saved.


### precision@k

In [14]:
top_k = test_df.sort_values('anomaly_score').head(100)  # lowest scores = most anomalous
caught_in_top_100 = top_k['is_scenario_day'].sum()
print(f"Scenario days caught in top 100 riskiest predictions: {caught_in_top_100} / 100")

top_k_500 = test_df.sort_values('anomaly_score').head(500)
caught_in_top_500 = top_k_500['is_scenario_day'].sum()
print(f"Scenario days caught in top 500 riskiest predictions: {caught_in_top_500} / 500")

Scenario days caught in top 100 riskiest predictions: 8 / 100
Scenario days caught in top 500 riskiest predictions: 25 / 500


### Conclusion
The good news: Scenario days are flagged at more than 3x the rate of normal days (6.0% vs 1.8%). This confirms there IS a real, learnable signal in your engineered features — the model isn't just guessing randomly. Your feature engineering (after-hours flags, USB deviations, rolling baselines) is genuinely contributing something.

The sobering news: Your baseline model still misses 94% of actual scenario days (only caught 82 out of 1,364). In SOC terms, this would be a very low recall — most real insider threat activity would slip past this model undetected.

Probable reasons :
1. A baseline AND an advanced model — Isolation Forest with default settings is meant to be your "floor," not your final answer
2. Threshold tuning — right now you used contamination=0.02 as a rough guess; this directly controls how many points get flagged, and it's worth testing other values
3. The autoencoder — better suited to learning complex, non-linear normal-behavior patterns than Isolation Forest's simpler splitting logic

## Precision@k
test set has 1,364 scenario days out of 225,281 total test user-days — a base rate of about 0.6%.
Random guessing would give you ~0.6 hits in a random sample of 100 (0.6%)
Our model gives you 8 hits in the top 100 (8%)

That's roughly 13x better than random chance. This is a meaningful, real signal — not noise.

### Precision@K Results (Isolation Forest Baseline)
- Precision@100: 8% (8/100) — vs. ~0.6% base rate, a ~13x improvement over random
- Precision@500: 5% (25/500)
- Interpretation: given a realistic analyst alert budget (e.g., reviewing 
  the top 100 riskiest user-days), this baseline model surfaces real 
  threat activity at a rate far above chance, though the majority of 
  alerts remain false positives — motivating both threshold refinement 
  and the more expressive autoencoder model.